# Data Cleaning

In [47]:
import numpy as np
import pandas as pd
import pymc as pm
import threadpoolctl
import pymc.sampling.mcmc
import arviz as az
import itertools

In [48]:
df = pd.read_csv('results.csv')

In [49]:
df['date'] = pd.to_datetime(df['date'])

In [50]:
modern_df = df[df['date'] >= '2018-01-01'].copy()

In [51]:
cleaned_df = modern_df[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'neutral']].copy()

In [52]:
cleaned_df = cleaned_df.dropna(subset=['home_score', 'away_score'])

In [53]:
cleaned_df['home_score'] = cleaned_df['home_score'].astype(int)
cleaned_df['away_score'] = cleaned_df['away_score'].astype(int)

# Add IDs and Weightings

In [54]:
teams = np.unique(cleaned_df[['home_team', 'away_team']].values)
num_teams = len(teams)

In [55]:
team_to_id = {team: i for i, team in enumerate(teams)}
id_to_team = {i: team for team, i in team_to_id.items()}

In [56]:
cleaned_df['home_team_id'] = cleaned_df['home_team'].map(team_to_id)
cleaned_df['away_team_id'] = cleaned_df['away_team'].map(team_to_id)

In [57]:
unique_tournaments = cleaned_df['tournament'].unique()

def calculate_prestige(name):
    n = name.lower()
    if n == 'fifa world cup' or 'cup of champions' in n: return 1.0
    if n in ['uefa euro', 'copa américa', 'african cup of nations', 'afc asian cup']: return 0.9
    if n in ['gold cup', 'oceania nations cup', 'arab cup', 'asean championship', 'aff championship', 'eaff championship', 'cafa nations cup']: return 0.8
    if 'qualification' in n: return 0.70
    if 'nations league' in n: return 0.60
    if any(x in n for x in ['conifa', 'games', 'island', 'vase', 'marianas']): return 0.15
    return 0.30

tournament_weights = {t: calculate_prestige(t) for t in unique_tournaments}
cleaned_df['tournament_weight'] = cleaned_df['tournament'].map(tournament_weights)

newest_date = cleaned_df['date'].max()
cleaned_df['days_ago'] = (newest_date - cleaned_df['date']).dt.days
cleaned_df['recency_weight'] = np.exp(-cleaned_df['days_ago'] / 730.0)

cleaned_df['final_weight'] = cleaned_df['recency_weight'] * cleaned_df['tournament_weight']

match_weights = cleaned_df['final_weight'].values / cleaned_df['final_weight'].max()

# Build Poisson Model

In [58]:
class DummyLimiter:
    def __init__(self, *args, **kwargs): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass

threadpoolctl.threadpool_limits = DummyLimiter
pymc.sampling.mcmc.threadpool_limits = DummyLimiter

In [59]:
home_team_ids = cleaned_df['home_team_id'].values
away_team_ids = cleaned_df['away_team_id'].values
home_goals = cleaned_df['home_score'].values
away_goals = cleaned_df['away_score'].values

neutral_mask = (~cleaned_df['neutral']).astype(int).values

with pm.Model() as poisson_model:
    intercept = pm.Normal('intercept', mu=0, sigma=1)
    home_adv = pm.Normal('home_adv', mu=0, sigma=1)
    
    sigma_att = pm.Exponential('sigma_att', lambda_=1.0)
    sigma_def = pm.Exponential('sigma_def', lambda_=1.0)

    atts_offset = pm.Normal('atts_offset', mu=0, sigma=1, shape=num_teams)
    defs_offset = pm.Normal('defs_offset', mu=0, sigma=1, shape=num_teams)
    
    atts_raw = atts_offset * sigma_att
    defs_raw = defs_offset * sigma_def
    
    atts = pm.Deterministic('atts', atts_raw - pm.math.mean(atts_raw))
    defs = pm.Deterministic('defs', defs_raw - pm.math.mean(defs_raw))
    
    home_theta = pm.math.exp(intercept + (home_adv * neutral_mask) + atts[home_team_ids] - defs[away_team_ids])
    away_theta = pm.math.exp(intercept + atts[away_team_ids] - defs[home_team_ids])
    
    home_dist = pm.Poisson.dist(mu=home_theta)
    away_dist = pm.Poisson.dist(mu=away_theta)
    
    pm.Potential('weighted_home_lik', match_weights * pm.logp(home_dist, home_goals))
    pm.Potential('weighted_away_lik', match_weights * pm.logp(away_dist, away_goals))

# Sample Probabilities

In [60]:
with poisson_model:
    trace_poisson = pm.sample(draws=1000, tune=1000, cores=1, return_inferencedata=True)

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [intercept, home_adv, sigma_att, sigma_def, atts_offset, defs_offset]


Output()

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 31 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


# Interpret

In [61]:
att_means = trace_poisson.posterior['atts'].mean(dim=['chain', 'draw']).values
def_means = trace_poisson.posterior['defs'].mean(dim=['chain', 'draw']).values

team_ratings = pd.DataFrame({
    'Team': [id_to_team[i] for i in range(num_teams)],
    'Attack': att_means,
    'Defense': def_means
})

In [62]:
def simulate_poisson_match(home_team, away_team, is_neutral=True):
    if home_team not in team_to_id or away_team not in team_to_id:
        print(f"Error: Make sure names exactly match dataset. Available teams include: {list(team_to_id.keys())[:5]}")
        return
        
    home_id = team_to_id[home_team]
    away_id = team_to_id[away_team]
    
    intercepts = trace_poisson.posterior['intercept'].values.flatten()
    home_advantages = trace_poisson.posterior['home_adv'].values.flatten()
    
    att_home = trace_poisson.posterior['atts'].values[:, :, home_id].flatten()
    def_home = trace_poisson.posterior['defs'].values[:, :, home_id].flatten()
    
    att_away = trace_poisson.posterior['atts'].values[:, :, away_id].flatten()
    def_away = trace_poisson.posterior['defs'].values[:, :, away_id].flatten()
    
    neutral_multiplier = 0 if is_neutral else 1
    
    home_theta = np.exp(intercepts + (home_advantages * neutral_multiplier) + att_home - def_away)
    away_theta = np.exp(intercepts + att_away - def_home)
    
    simulated_home_goals = np.random.poisson(home_theta)
    simulated_away_goals = np.random.poisson(away_theta)
    
    total_simulations = len(simulated_home_goals)
    home_wins = np.sum(simulated_home_goals > simulated_away_goals)
    away_wins = np.sum(simulated_away_goals > simulated_home_goals)
    draws = np.sum(simulated_home_goals == simulated_away_goals)
    
    print(f"=== MATCH SIMULATION: {home_team} vs {away_team} ===")
    print(f"Venue: {'Neutral Ground' if is_neutral else 'Home Match for ' + home_team}\n")
    print(f"{home_team} Win Probability: {home_wins / total_simulations * 100:.2f}%")
    print(f"{away_team} Win Probability: {away_wins / total_simulations * 100:.2f}%")
    print(f"Draw Probability: {draws / total_simulations * 100:.2f}%")
    print(f"Expected Average Scoreline: {simulated_home_goals.mean():.2f} - {simulated_away_goals.mean():.2f}")
    
    return simulated_home_goals, simulated_away_goals

In [63]:
summary_stats = az.summary(trace_poisson, var_names=['intercept', 'home_adv', 'sigma_att', 'sigma_def'])
print(summary_stats)

            mean      sd eti89_lb eti89_ub  ess_bulk  ess_tail r_hat  \
intercept  0.154  0.0308     0.11      0.2      2078      1423  1.00   
home_adv   0.286   0.038     0.23     0.35      3102      1612  1.00   
sigma_att  0.333    0.03     0.29     0.38       700       981  1.01   
sigma_def  0.353   0.031     0.31     0.41       669       870  1.01   

          mcse_mean  mcse_sd  
intercept   0.00067  0.00048  
home_adv    0.00068   0.0005  
sigma_att    0.0011  0.00084  
sigma_def    0.0012  0.00085  


In [64]:
print("\n--- TOP 10 ATTACKING TEAMS ---")
print(team_ratings.sort_values(by='Attack', ascending=False).head(10)[['Team', 'Attack']].to_string(index=False))

print("\n--- TOP 10 DEFENDING TEAMS ---")
print(team_ratings.sort_values(by='Defense', ascending=False).head(10)[['Team', 'Defense']].to_string(index=False))


--- TOP 10 ATTACKING TEAMS ---
       Team   Attack
    Germany 0.657908
      Spain 0.640578
Netherlands 0.624547
     Norway 0.614641
     France 0.596934
      Japan 0.592934
   Portugal 0.572837
  Argentina 0.560882
    Senegal 0.502823
    Belgium 0.495589

--- TOP 10 DEFENDING TEAMS ---
     Team  Defense
  Morocco 0.686014
Argentina 0.676630
  Ecuador 0.601169
  England 0.586842
    Spain 0.578337
   Mexico 0.564812
   Brazil 0.493425
    Japan 0.485339
 DR Congo 0.485247
   France 0.481454


# Build Negative Binomial Model

In [65]:
with pm.Model() as nb_model:
    intercept = pm.Normal('intercept', mu=0, sigma=1)
    home_adv = pm.Normal('home_adv', mu=0, sigma=1)
    
    sigma_att = pm.Exponential('sigma_att', lambda_=1.0)
    sigma_def = pm.Exponential('sigma_def', lambda_=1.0)
    
    alpha_home = pm.Exponential('alpha_home', lambda_=1.0)
    alpha_away = pm.Exponential('alpha_away', lambda_=1.0)
    
    atts_offset = pm.Normal('atts_offset', mu=0, sigma=1, shape=num_teams)
    defs_offset = pm.Normal('defs_offset', mu=0, sigma=1, shape=num_teams)
    
    atts_raw = atts_offset * sigma_att
    defs_raw = defs_offset * sigma_def
    
    atts = pm.Deterministic('atts', atts_raw - pm.math.mean(atts_raw))
    defs = pm.Deterministic('defs', defs_raw - pm.math.mean(defs_raw))
    
    home_theta = pm.math.exp(intercept + (home_adv * neutral_mask) + atts[home_team_ids] - defs[away_team_ids])
    away_theta = pm.math.exp(intercept + atts[away_team_ids] - defs[home_team_ids])
    
    home_dist = pm.NegativeBinomial.dist(mu=home_theta, alpha=alpha_home)
    away_dist = pm.NegativeBinomial.dist(mu=away_theta, alpha=alpha_away)
    
    pm.Potential('weighted_home_lik', match_weights * pm.logp(home_dist, home_goals))
    pm.Potential('weighted_away_lik', match_weights * pm.logp(away_dist, away_goals))

In [66]:
with nb_model:
    trace_nb = pm.sample(draws=1000, tune=1000, cores=1, return_inferencedata=True)

Initializing NUTS using jitter+adapt_diag...
C:\Users\Oscar2\anaconda3\envs\wc_env\Lib\site-packages\pytensor\tensor\rewriting\elemwise.py:1034: UserWarning: Loop fusion failed because the resulting node would exceed the kernel argument limit.
  warn(
Sequential sampling (2 chains in 1 job)
NUTS: [intercept, home_adv, sigma_att, sigma_def, alpha_home, alpha_away, atts_offset, defs_offset]


Output()

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 267 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


In [67]:
trace_nb.to_netcdf('nb_trace.nc')

In [68]:
att_means = trace_nb.posterior['atts'].mean(dim=['chain', 'draw']).values
def_means = trace_nb.posterior['defs'].mean(dim=['chain', 'draw']).values

team_ratings = pd.DataFrame({
    'Team': [id_to_team[i] for i in range(num_teams)],
    'Attack': att_means,
    'Defense': def_means
})

In [69]:
def simulate_nb_match(home_team, away_team, is_neutral=True):
    if home_team not in team_to_id or away_team not in team_to_id:
        print(f"Error: One or both teams not found in dataset mapping.")
        return
        
    home_id = team_to_id[home_team]
    away_id = team_to_id[away_team]
    
    intercepts = trace_nb.posterior['intercept'].values.flatten()
    home_advantages = trace_nb.posterior['home_adv'].values.flatten()
    alphas_home = trace_nb.posterior['alpha_home'].values.flatten()
    alphas_away = trace_nb.posterior['alpha_away'].values.flatten()
    
    att_home = trace_nb.posterior['atts'].values[:, :, home_id].flatten()
    def_home = trace_nb.posterior['defs'].values[:, :, home_id].flatten()
    att_away = trace_nb.posterior['atts'].values[:, :, away_id].flatten()
    def_away = trace_nb.posterior['defs'].values[:, :, away_id].flatten()
    
    neutral_multiplier = 0 if is_neutral else 1
    
    home_theta = np.exp(intercepts + (home_advantages * neutral_multiplier) + att_home - def_away)
    away_theta = np.exp(intercepts + att_away - def_home)
    
    p_home = alphas_home / (alphas_home + home_theta)
    p_away = alphas_away / (alphas_away + away_theta)
    
    sim_home_goals = np.random.negative_binomial(n=alphas_home, p=p_home)
    sim_away_goals = np.random.negative_binomial(n=alphas_away, p=p_away)
    
    total_simulations = len(sim_home_goals)
    home_wins = np.sum(sim_home_goals > sim_away_goals)
    away_wins = np.sum(sim_away_goals > sim_home_goals)
    draws = np.sum(sim_sim_goals == sim_away_goals) if 'sim_sim_goals' in locals() else np.sum(sim_home_goals == sim_away_goals)
    
    print(f"\n=== MATCH SIMULATION: {home_team} vs {away_team} ===")
    print(f"Venue: {'Neutral Ground' if is_neutral else 'Home Match for ' + home_team}\n")
    print(f"{home_team} Win Probability: {home_wins / total_simulations * 100:.2f}%")
    print(f"{away_team} Win Probability: {away_wins / total_simulations * 100:.2f}%")
    print(f"Draw Probability: {draws / total_simulations * 100:.2f}%")
    print(f"Expected Average Scoreline: {sim_home_goals.mean():.2f} - {sim_away_goals.mean():.2f}\n")
    
    score_matrix = np.zeros((4, 4))
    for h in range(4):
        for a in range(4):
            match_count = np.sum((sim_home_goals == h) & (sim_away_goals == a))
            score_matrix[h, a] = (match_count / total_simulations) * 100
            
    matrix_df = pd.DataFrame(
        score_matrix, 
        index=[f"{home_team} {i}" for i in range(4)], 
        columns=[f"{away_team} {j}" for j in range(4)]
    )
    print("--- EXACT SCORELINE COMBINATION PROBABILITIES (< 4 GOALS) ---")
    print(matrix_df.round(2).to_string())
    
    return sim_home_goals, sim_away_goals

In [70]:
print("\n--- GLOBAL PARAMETER DIAGNOSTICS ---")
print(az.summary(trace_nb, var_names=['intercept', 'home_adv', 'alpha_home', 'alpha_away']))

print("\n--- TOP 10 ATTACKING TEAMS ---")
print(team_ratings.sort_values(by='Attack', ascending=False).head(10)[['Team', 'Attack']].to_string(index=False))

print("\n--- TOP 10 DEFENDING TEAMS ---")
print(team_ratings.sort_values(by='Defense', ascending=False).head(10)[['Team', 'Defense']].to_string(index=False))


--- GLOBAL PARAMETER DIAGNOSTICS ---
             mean     sd eti89_lb eti89_ub  ess_bulk  ess_tail r_hat  \
intercept    0.17  0.032     0.12     0.22      2090      1374  1.00   
home_adv    0.306  0.045     0.23     0.38      2650      1478  1.00   
alpha_home   4.82   0.87      3.6      6.3      1944      1337  1.00   
alpha_away   4.01   0.77      2.9      5.3      2101      1660  1.00   

           mcse_mean  mcse_sd  
intercept     0.0007   0.0005  
home_adv     0.00087  0.00061  
alpha_home     0.021    0.018  
alpha_away     0.017    0.014  

--- TOP 10 ATTACKING TEAMS ---
       Team   Attack
Netherlands 0.463796
      Spain 0.458394
    Germany 0.456348
     Norway 0.433645
     France 0.427401
      Japan 0.416091
   Portugal 0.398026
  Argentina 0.375537
    Senegal 0.369340
    England 0.348746

--- TOP 10 DEFENDING TEAMS ---
       Team  Defense
    Morocco 0.507266
  Argentina 0.444408
    England 0.411793
    Ecuador 0.391230
      Spain 0.390954
     Mexico 0.382256

In [71]:
def compare_models(home_team, away_team, is_neutral=True):
    if home_team not in team_to_id or away_team not in team_to_id:
        print("Error: One or both teams not found.")
        return
        
    home_id = team_to_id[home_team]
    away_id = team_to_id[away_team]
    neutral_multiplier = 0 if is_neutral else 1
    
    p_intercepts = trace_poisson.posterior['intercept'].values.flatten()
    p_home_advs = trace_poisson.posterior['home_adv'].values.flatten()
    p_att_home = trace_poisson.posterior['atts'].values[:, :, home_id].flatten()
    p_def_home = trace_poisson.posterior['defs'].values[:, :, home_id].flatten()
    p_att_away = trace_poisson.posterior['atts'].values[:, :, away_id].flatten()
    p_def_away = trace_poisson.posterior['defs'].values[:, :, away_id].flatten()
    
    p_home_theta = np.exp(p_intercepts + (p_home_advs * neutral_multiplier) + p_att_home - p_def_away)
    p_away_theta = np.exp(p_intercepts + p_att_away - p_def_home)
    
    sim_p_home = np.random.poisson(p_home_theta)
    sim_p_away = np.random.poisson(p_away_theta)
    
    nb_intercepts = trace_nb.posterior['intercept'].values.flatten()
    nb_home_advs = trace_nb.posterior['home_adv'].values.flatten()
    alphas_home = trace_nb.posterior['alpha_home'].values.flatten()
    alphas_away = trace_nb.posterior['alpha_away'].values.flatten()
    
    nb_att_home = trace_nb.posterior['atts'].values[:, :, home_id].flatten()
    nb_def_home = trace_nb.posterior['defs'].values[:, :, home_id].flatten()
    nb_att_away = trace_nb.posterior['atts'].values[:, :, away_id].flatten()
    nb_def_away = trace_nb.posterior['defs'].values[:, :, nb_def_away_id if 'nb_def_away_id' in locals() else away_id].flatten()
    
    nb_home_theta = np.exp(nb_intercepts + (nb_home_advs * neutral_multiplier) + nb_att_home - nb_def_away)
    nb_away_theta = np.exp(nb_intercepts + nb_att_away - nb_def_home)
    
    p_nb_home = alphas_home / (alphas_home + nb_home_theta)
    p_nb_away = alphas_away / (alphas_away + nb_away_theta)
    
    sim_nb_home = np.random.negative_binomial(n=alphas_home, p=p_nb_home)
    sim_nb_away = np.random.negative_binomial(n=alphas_away, p=p_nb_away)
    
    total_sims = len(sim_p_home)
    
    pct_p_hw, pct_nb_hw = np.sum(sim_p_home > sim_p_away)/total_sims, np.sum(sim_nb_home > sim_nb_away)/total_sims
    pct_p_aw, pct_nb_aw = np.sum(sim_p_away > sim_p_home)/total_sims, np.sum(sim_nb_away > sim_nb_home)/total_sims
    pct_p_d, pct_nb_d = np.sum(sim_p_home == sim_p_away)/total_sims, np.sum(sim_nb_home == sim_nb_away)/total_sims
    
    print(f"===========================================================")
    print(f"   HEAD-TO-HEAD MODEL COMPARISON: {home_team} vs {away_team}")
    print(f"===========================================================")
    print(f"METRIC                      | POISSON MODEL | NEG BINOMIAL MODEL")
    print(f"-----------------------------------------------------------")
    print(f"{home_team:27} Win % | {pct_p_hw*100:12.2f}% | {pct_nb_hw*100:17.2f}%")
    print(f"{away_team:27} Win % | {pct_p_aw*100:12.2f}% | {pct_nb_aw*100:17.2f}%")
    print(f"Draw %                      | {pct_p_d*100:12.2f}% | {pct_nb_d*100:17.2f}%")
    print(f"Expected Avg Goals ({home_team})| {sim_p_home.mean():13.2f}  | {sim_nb_home.mean():18.2f}")
    print(f"Expected Avg Goals ({away_team:11})| {sim_p_away.mean():13.2f}  | {sim_nb_away.mean():18.2f}")
    print(f"-----------------------------------------------------------")
    
    matrix_p = np.zeros((4, 4))
    matrix_nb = np.zeros((4, 4))
    for h in range(4):
        for a in range(4):
            matrix_p[h, a] = (np.sum((sim_p_home == h) & (sim_p_away == a)) / total_sims) * 100
            matrix_nb[h, a] = (np.sum((sim_nb_home == h) & (sim_nb_away == a)) / total_sims) * 100
            
    print(f"\n--- POISSON SCORELINE COMBINATIONS (< 4 GOALS) ---")
    df_p = pd.DataFrame(matrix_p, index=[f"{home_team} {i}" for i in range(4)], columns=[f"{away_team} {j}" for j in range(4)])
    print(df_p.round(2).to_string())
    
    print(f"\n--- NEGATIVE BINOMIAL SCORELINE COMBINATIONS (< 4 GOALS) ---")
    df_nb = pd.DataFrame(matrix_nb, index=[f"{home_team} {i}" for i in range(4)], columns=[f"{away_team} {j}" for j in range(4)])
    print(df_nb.round(2).to_string())

    return

In [72]:
compare_models("England", "DR Congo", is_neutral=True)

   HEAD-TO-HEAD MODEL COMPARISON: England vs DR Congo
METRIC                      | POISSON MODEL | NEG BINOMIAL MODEL
-----------------------------------------------------------
England                     Win % |        48.60% |             43.25%
DR Congo                    Win % |        21.75% |             26.65%
Draw %                      |        29.65% |             30.10%
Expected Avg Goals (England)|          1.22  |               1.26
Expected Avg Goals (DR Congo   )|          0.71  |               0.89
-----------------------------------------------------------

--- POISSON SCORELINE COMBINATIONS (< 4 GOALS) ---
           DR Congo 0  DR Congo 1  DR Congo 2  DR Congo 3
England 0       15.30        9.65        3.35        1.40
England 1       17.85       11.55        5.40        0.80
England 2       10.75        7.55        2.55        0.55
England 3        4.45        2.90        1.00        0.25

--- NEGATIVE BINOMIAL SCORELINE COMBINATIONS (< 4 GOALS) ---
           DR 

# Tournament Simulation

## Setup

In [74]:
def generate_penalty_probabilities(csv_path='shootouts.csv', start_year=2016, alpha=4):
    """
    Cleans, filters, and applies Bayesian Smoothing (Laplace) to international 
    shootout data to output simulation-ready win probabilities.
    """
    df = pd.read_csv(csv_path)
    df['date'] = pd.to_datetime(df['date'])
    df_filtered = df[df['date'] >= f'{start_year}-01-01']
    
    team_stats = {}
    
    for _, row in df_filtered.iterrows():
        home = row['home_team']
        away = row['away_team']
        winner = row['winner']
        
        for team in [home, away]:
            if team not in team_stats:
                team_stats[team] = {'wins': 0, 'total': 0}
                
        team_stats[home]['total'] += 1
        team_stats[away]['total'] += 1
        
        if winner == home:
            team_stats[home]['wins'] += 1
        elif winner == away:
            team_stats[away]['wins'] += 1

    penalty_probs = {}
    for team, stats in team_stats.items():
        wins = stats['wins']
        total = stats['total']
        smoothed_prob = (wins + alpha) / (total + (2 * alpha))
        penalty_probs[team] = round(smoothed_prob, 4)
        
    return penalty_probs

PENALTY_PROBS = generate_penalty_probabilities('shootouts.csv', start_year=2016, alpha=4)

def get_shootout_win_prob(team_name):
    """Returns the smoothed win probability or a 50% coin-flip baseline if missing."""
    return PENALTY_PROBS.get(team_name, 0.5000)

In [75]:
CURRENT_MATCH_LOG = []

def run_silent_match(home_team, away_team, is_knockout=False, is_neutral=True, stage="Group"):
    """Runs a single match simulation, logs telemetry, and returns match outcomes."""
    global CURRENT_MATCH_LOG
    
    home_id = team_to_id[home_team]
    away_id = team_to_id[away_team]
    
    chain_idx = np.random.randint(0, trace_nb.posterior.sizes['chain'])
    draw_idx = np.random.randint(0, trace_nb.posterior.sizes['draw'])
    
    intercept = trace_nb.posterior['intercept'].values[chain_idx, draw_idx]
    home_adv = trace_nb.posterior['home_adv'].values[chain_idx, draw_idx]
    alpha_h = trace_nb.posterior['alpha_home'].values[chain_idx, draw_idx]
    alpha_a = trace_nb.posterior['alpha_away'].values[chain_idx, draw_idx]
    
    att_h = trace_nb.posterior['atts'].values[chain_idx, draw_idx, home_id]
    def_h = trace_nb.posterior['defs'].values[chain_idx, draw_idx, home_id]
    att_a = trace_nb.posterior['atts'].values[chain_idx, draw_idx, away_id]
    def_a = trace_nb.posterior['defs'].values[chain_idx, draw_idx, away_id]
    
    neutral_multiplier = 0 if is_neutral else 1
    
    home_theta = np.exp(intercept + (home_adv * neutral_multiplier) + att_h - def_a)
    away_theta = np.exp(intercept + att_a - def_h)
    
    p_h = alpha_h / (alpha_h + home_theta)
    p_a = alpha_a / (alpha_a + away_theta)
    
    h_goals = np.random.negative_binomial(n=alpha_h, p=p_h)
    a_goals = np.random.negative_binomial(n=alpha_a, p=p_a)
    
    method = "90 Mins"
    if h_goals > a_goals:
        winner = home_team
    elif a_goals > h_goals:
        winner = away_team
    else:
        if is_knockout:
            prob_h = get_shootout_win_prob(home_team)
            prob_a = get_shootout_win_prob(away_team)
            relative_p_h = prob_h / (prob_h + prob_a)
            
            if np.random.random() < relative_p_h:
                winner = home_team
            else:
                winner = away_team
            method = "Penalties"
        else:
            winner = "Draw"
            
    CURRENT_MATCH_LOG.append({
        'stage': stage,
        'home': home_team,
        'away': away_team,
        'h_goals': int(h_goals),
        'a_goals': int(a_goals),
        'winner': winner,
        'home_theta': float(home_theta),
        'away_theta': float(away_theta),
        'method': method
    })
    
    return h_goals, a_goals, winner

## Complete Simulation (Groups + Knockouts)

In [76]:
tournament_groups_fixed = {
    'Group A': ['Mexico', 'South Africa', 'South Korea', 'Czech Republic'],
    'Group B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'Group C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'Group D': ['United States', 'Paraguay', 'Australia', 'Turkey'],
    'Group E': ['Germany', 'Curaçao', 'Ivory Coast', 'Ecuador'],
    'Group F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'Group G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'Group H': ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay'],
    'Group I': ['France', 'Senegal', 'Iraq', 'Norway'],
    'Group J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'Group K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'],
    'Group L': ['England', 'Croatia', 'Ghana', 'Panama'],
}

def simulate_full_tournament(groups_dict):
    """Simulates a complete World Cup tournament and relies on global telemetry logging."""
    winners = {}
    runners_up = {}
    all_3rd = []
    
    for group_name, team_list in groups_dict.items():
        standings = pd.DataFrame(0, index=team_list, columns=['Points', 'GD', 'GS'])
        group_letter = group_name.split()[-1]
        
        for i in range(len(team_list)):
            for j in range(i + 1, len(team_list)):
                t1, t2 = team_list[i], team_list[j]
                g1, g2, winner = run_silent_match(t1, t2, is_knockout=False, is_neutral=True, stage="Group")
                
                standings.loc[t1, 'GS'] += g1
                standings.loc[t2, 'GS'] += g2
                standings.loc[t1, 'GD'] += (g1 - g2)
                standings.loc[t2, 'GD'] += (g2 - g1)
                
                if winner == t1: standings.loc[t1, 'Points'] += 3
                elif winner == t2: standings.loc[t2, 'Points'] += 3
                else:
                    standings.loc[t1, 'Points'] += 1
                    standings.loc[t2, 'Points'] += 1
                    
        sorted_s = standings.sort_values(by=['Points', 'GD', 'GS'], ascending=False)
        winners[group_letter] = sorted_s.index[0]
        runners_up[group_letter] = sorted_s.index[1]
        
        third_team = sorted_s.index[2]
        all_3rd.append({
            'Team': third_team, 'Group': group_letter,
            'Points': sorted_s.loc[third_team, 'Points'],
            'GD': sorted_s.loc[third_team, 'GD'],
            'GS': sorted_s.loc[third_team, 'GS']
        })
        
    third_df = pd.DataFrame(all_3rd).sort_values(by=['Points', 'GD', 'GS'], ascending=False)
    top_8_wildcards = third_df.head(8).to_dict('records')
    
    slot_opponents = {74: 'E', 77: 'I', 79: 'A', 80: 'L', 81: 'D', 82: 'G', 85: 'B', 87: 'K'}
    slots_list = [74, 77, 79, 80, 81, 82, 85, 87]
    
    def match_slots(slot_idx, current_assignment, available):
        if slot_idx == len(slots_list): return current_assignment
        slot_id = slots_list[slot_idx]
        opp_group = slot_opponents[slot_id]
        for i, wc in enumerate(available):
            if wc['Group'] != opp_group:
                res = match_slots(slot_idx + 1, {**current_assignment, slot_id: wc['Team']}, available[:i] + available[i+1:])
                if res: return res
        return None

    assigned_wildcards = match_slots(0, {}, top_8_wildcards)
    if not assigned_wildcards:
        assigned_wildcards = {slot_id: top_8_wildcards[i]['Team'] for i, slot_id in enumerate(slots_list)}
        
    r32 = {}
    r32[73] = run_silent_match(runners_up['A'], runners_up['B'], is_knockout=True, stage="R32")[2]
    r32[74] = run_silent_match(winners['E'], assigned_wildcards[74], is_knockout=True, stage="R32")[2]
    r32[75] = run_silent_match(winners['F'], runners_up['C'], is_knockout=True, stage="R32")[2]
    r32[76] = run_silent_match(winners['C'], runners_up['F'], is_knockout=True, stage="R32")[2]
    r32[77] = run_silent_match(winners['I'], assigned_wildcards[77], is_knockout=True, stage="R32")[2]
    r32[78] = run_silent_match(runners_up['E'], runners_up['I'], is_knockout=True, stage="R32")[2]
    r32[79] = run_silent_match(winners['A'], assigned_wildcards[79], is_knockout=True, stage="R32")[2]
    r32[80] = run_silent_match(winners['L'], assigned_wildcards[80], is_knockout=True, stage="R32")[2]
    r32[81] = run_silent_match(winners['D'], assigned_wildcards[81], is_knockout=True, stage="R32")[2]
    r32[82] = run_silent_match(winners['G'], assigned_wildcards[82], is_knockout=True, stage="R32")[2]
    r32[83] = run_silent_match(runners_up['K'], runners_up['L'], is_knockout=True, stage="R32")[2]
    r32[84] = run_silent_match(winners['H'], runners_up['J'], is_knockout=True, stage="R32")[2]
    r32[85] = run_silent_match(winners['B'], assigned_wildcards[85], is_knockout=True, stage="R32")[2]
    r32[86] = run_silent_match(winners['J'], runners_up['H'], is_knockout=True, stage="R32")[2]
    r32[87] = run_silent_match(winners['K'], assigned_wildcards[87], is_knockout=True, stage="R32")[2]
    r32[88] = run_silent_match(runners_up['D'], runners_up['G'], is_knockout=True, stage="R32")[2]

    r16 = {}
    r16[89] = run_silent_match(r32[73], r32[74], is_knockout=True, stage="R16")[2]
    r16[90] = run_silent_match(r32[75], r32[77], is_knockout=True, stage="R16")[2]
    r16[91] = run_silent_match(r32[76], r32[78], is_knockout=True, stage="R16")[2]
    r16[92] = run_silent_match(r32[79], r32[80], is_knockout=True, stage="R16")[2]
    r16[93] = run_silent_match(r32[83], r32[84], is_knockout=True, stage="R16")[2]
    r16[94] = run_silent_match(r32[81], r32[82], is_knockout=True, stage="R16")[2]
    r16[95] = run_silent_match(r32[86], r32[88], is_knockout=True, stage="R16")[2]
    r16[96] = run_silent_match(r32[85], r32[87], is_knockout=True, stage="R16")[2]

    qf = {}
    qf[97] = run_silent_match(r16[89], r16[90], is_knockout=True, stage="QF")[2]
    qf[98] = run_silent_match(r16[93], r16[94], is_knockout=True, stage="QF")[2]
    qf[99] = run_silent_match(r16[91], r16[92], is_knockout=True, stage="QF")[2]
    qf[100] = run_silent_match(r16[95], r16[96], is_knockout=True, stage="QF")[2]

    sf = {}
    sf[101] = run_silent_match(qf[97], qf[98], is_knockout=True, stage="SF")[2]
    sf[102] = run_silent_match(qf[99], qf[100], is_knockout=True, stage="SF")[2]

    champion = run_silent_match(sf[101], sf[102], is_knockout=True, stage="Final")[2]
    return champion

In [77]:
all_teams = list(set(itertools.chain(*tournament_groups_fixed.values())))

survival_matrix = {team: {'Group Exit': 0, 'R32': 0, 'R16': 0, 'QF': 0, 'SF': 0, 'Runner-up': 0, 'Champion': 0} for team in all_teams}

total_over_6_games = 0
max_over_6_in_single_iter = 0
absolute_highest_scoring_game = {'goals': -1, 'match_details': ""}

total_upsets_count = 0
max_upsets_in_single_iter = 0
peak_upset_iteration_index = -1
peak_iteration_upset_matches_list = []

UPSET_RATIO_THRESHOLD = 1.75

print("Running 1,000 deep-insight tournament iterations...")

for iteration in range(1000):
    CURRENT_MATCH_LOG = []
    
    champ = simulate_full_tournament(tournament_groups_fixed)
    
    iter_over_6_count = 0
    iter_upset_list = []
    
    iteration_team_ceilings = {team: 'Group Exit' for team in all_teams}
    
    for match in CURRENT_MATCH_LOG:
        total_goals = match['h_goals'] + match['a_goals']
        
        if total_goals > 6:
            iter_over_6_count += 1
            if total_goals > absolute_highest_scoring_game['goals']:
                absolute_highest_scoring_game['goals'] = total_goals
                absolute_highest_scoring_game['match_details'] = f"{match['home']} {match['h_goals']}-{match['a_goals']} {match['away']} ({match['stage']})"
        
        theta_ratio = match['home_theta'] / match['away_theta']
        is_upset = False
        if theta_ratio >= UPSET_RATIO_THRESHOLD and match['winner'] == match['away']:
            is_upset = True
        elif theta_ratio <= (1 / UPSET_RATIO_THRESHOLD) and match['winner'] == match['home']:
            is_upset = True
            
        if is_upset:
            iter_upset_list.append(f"{match['home']} vs {match['away']} ({match['h_goals']}-{match['a_goals']}, Winner: {match['winner']} at {match['stage']})")
            
        if match['stage'] != 'Group':
            loser = match['away'] if match['winner'] == match['home'] else match['home']
            if match['stage'] == 'R32': iteration_team_ceilings[loser] = 'R32'
            elif match['stage'] == 'R16': iteration_team_ceilings[loser] = 'R16'
            elif match['stage'] == 'QF': iteration_team_ceilings[loser] = 'QF'
            elif match['stage'] == 'SF': iteration_team_ceilings[loser] = 'SF'
            elif match['stage'] == 'Final': 
                iteration_team_ceilings[loser] = 'Runner-up'
                iteration_team_ceilings[match['winner']] = 'Champion'

    for team, stage in iteration_team_ceilings.items():
        survival_matrix[team][stage] += 1
        
    total_over_6_games += iter_over_6_count
    if iter_over_6_count > max_over_6_in_single_iter:
        max_over_6_in_single_iter = iter_over_6_count
        
    total_upsets_count += len(iter_upset_list)
    if len(iter_upset_list) > max_upsets_in_single_iter:
        max_upsets_in_single_iter = len(iter_upset_list)
        peak_upset_iteration_index = iteration
        peak_iteration_upset_matches_list = iter_upset_list

Running 1,000 deep-insight tournament iterations...


In [78]:
survival_df = pd.DataFrame.from_dict(survival_matrix, orient='index')
columns_order = ['Group Exit', 'R32', 'R16', 'QF', 'SF', 'Runner-up', 'Champion']
survival_df = survival_df[columns_order].sort_values(by='Champion', ascending=False)

print("\n=======================================================")
print("             TEAM STAGE SURVIVAL MATRIX                ")
print("=======================================================")
print(survival_df.to_string())


             TEAM STAGE SURVIVAL MATRIX                
                        Group Exit  R32  R16   QF  SF  Runner-up  Champion
England                        173  353  200  118  63         32        61
Argentina                      197  314  200  111  78         41        59
Morocco                        159  354  194  119  73         43        58
Spain                          117  378  228  102  76         45        54
Portugal                       212  325  194  117  59         50        43
Mexico                         230  338  189  104  60         36        43
Japan                          178  368  193  131  56         33        41
Belgium                        232  343  197  100  50         37        41
Germany                        223  345  189  114  61         30        38
Netherlands                    229  368  172   97  65         31        38
Senegal                        246  334  199   88  65         31        37
Colombia                       311  306  18

In [79]:
print("\n=======================================================")
print("            HIGH-SCORING THRILLER LOGS                 ")
print("=======================================================")
print(f"Total matches exceeding 6 goals: {total_over_6_games}")
print(f"Max high-scoring matches in a single tournament: {max_over_6_in_single_iter}")
print(f"Highest scoring single game recorded: {absolute_highest_scoring_game['match_details']}")

print("\n=======================================================")
print("             DYNAMIC TOURNAMENT UPSETS                 ")
print("=======================================================")
print(f"Total model-driven upsets across all iterations: {total_upsets_count}")
print(f"Most chaotic iteration index: #{peak_upset_iteration_index} (with {max_upsets_in_single_iter} upsets)")
print("\nUpset match logs from that chaotic iteration:")
for log in peak_iteration_upset_matches_list:
    print(f" - {log}")


            HIGH-SCORING THRILLER LOGS                 
Total matches exceeding 6 goals: 3857
Max high-scoring matches in a single tournament: 11
Highest scoring single game recorded: Qatar 1-18 Switzerland (Group)

             DYNAMIC TOURNAMENT UPSETS                 
Total model-driven upsets across all iterations: 6625
Most chaotic iteration index: #964 (with 17 upsets)

Upset match logs from that chaotic iteration:
 - Bosnia and Herzegovina vs Switzerland (1-0, Winner: Bosnia and Herzegovina at Group)
 - Qatar vs Switzerland (1-0, Winner: Qatar at Group)
 - Morocco vs Haiti (0-1, Winner: Haiti at Group)
 - Paraguay vs Turkey (3-2, Winner: Paraguay at Group)
 - Belgium vs Egypt (0-1, Winner: Egypt at Group)
 - Spain vs Cape Verde (0-1, Winner: Cape Verde at Group)
 - France vs Senegal (1-3, Winner: Senegal at Group)
 - Argentina vs Austria (0-1, Winner: Austria at Group)
 - Algeria vs Austria (1-2, Winner: Austria at Group)
 - South Korea vs Qatar (4-7, Winner: Qatar at R32)
 - E

## Knockouts Simulation

In [80]:
real_world_matchups = [
    ('Germany', 'Paraguay'), ('France', 'Sweden'), ('South Africa', 'Canada'), ('Netherlands', 'Morocco'),
    ('Portugal', 'Croatia'), ('Spain', 'Austria'), ('United States', 'Bosnia and Herzegovina'), ('Belgium', 'Senegal'),
    ('Brazil', 'Japan'), ('Ivory Coast', 'Norway'), ('Mexico', 'Ecuador'), ('England', 'DR Congo'),
    ('Argentina', 'Cape Verde'), ('Australia', 'Egypt'), ('Switzerland', 'Algeria'), ('Colombia', 'Ghana')
]

CURRENT_MATCH_LOG = []

def simulate_knockouts(bracket_matchups, iterations=1000):
    """Simulates a standalone knockout bracket and gathers deep telemetry."""
    global CURRENT_MATCH_LOG
    
    all_knockout_teams = list(set([team for matchup in bracket_matchups for team in matchup]))
    
    survival_matrix = {team: {'R32': 0, 'R16': 0, 'QF': 0, 'SF': 0, 'Runner-up': 0, 'Champion': 0} for team in all_knockout_teams}
    total_over_6_games = 0
    max_over_6_in_single_iter = 0
    absolute_highest_scoring_game = {'goals': -1, 'match_details': ""}
    
    total_upsets_count = 0
    max_upsets_in_single_iter = 0
    peak_upset_iteration_index = -1
    peak_iteration_upset_matches_list = []
    
    UPSET_RATIO_THRESHOLD = 1.75
    
    for iteration in range(iterations):
        CURRENT_MATCH_LOG = []
        
        r16_teams = []
        for team_a, team_b in bracket_matchups:
            _, _, winner = run_silent_match(team_a, team_b, is_knockout=True, is_neutral=True, stage="R32")
            r16_teams.append(winner)
            
        qf_teams = []
        for i in range(0, len(r16_teams), 2):
            _, _, winner = run_silent_match(r16_teams[i], r16_teams[i+1], is_knockout=True, is_neutral=True, stage="R16")
            qf_teams.append(winner)
            
        sf_teams = []
        for i in range(0, len(qf_teams), 2):
            _, _, winner = run_silent_match(qf_teams[i], qf_teams[i+1], is_knockout=True, is_neutral=True, stage="QF")
            sf_teams.append(winner)
            
        finalists = []
        for i in range(0, len(sf_teams), 2):
            _, _, winner = run_silent_match(sf_teams[i], sf_teams[i+1], is_knockout=True, is_neutral=True, stage="SF")
            finalists.append(winner)
            
        _, _, champion = run_silent_match(finalists[0], finalists[1], is_knockout=True, is_neutral=True, stage="Final")
        
        iter_over_6_count = 0
        iter_upset_list = []
        
        iteration_team_ceilings = {team: 'R32' for team in all_knockout_teams}
        
        for match in CURRENT_MATCH_LOG:
            total_goals = match['h_goals'] + match['a_goals']
            
            if total_goals > 6:
                iter_over_6_count += 1
                if total_goals > absolute_highest_scoring_game['goals']:
                    absolute_highest_scoring_game['goals'] = total_goals
                    absolute_highest_scoring_game['match_details'] = f"{match['home']} {match['h_goals']}-{match['a_goals']} {match['away']} ({match['stage']})"
            
            theta_ratio = match['home_theta'] / match['away_theta']
            is_upset = False
            if theta_ratio >= UPSET_RATIO_THRESHOLD and match['winner'] == match['away']:
                is_upset = True
            elif theta_ratio <= (1 / UPSET_RATIO_THRESHOLD) and match['winner'] == match['home']:
                is_upset = True
                
            if is_upset:
                iter_upset_list.append(f"{match['home']} vs {match['away']} ({match['h_goals']}-{match['a_goals']}, Winner: {match['winner']} at {match['stage']})")
            
            loser = match['away'] if match['winner'] == match['home'] else match['home']
            if match['stage'] == 'R32': iteration_team_ceilings[loser] = 'R32'
            elif match['stage'] == 'R16': iteration_team_ceilings[loser] = 'R16'
            elif match['stage'] == 'QF': iteration_team_ceilings[loser] = 'QF'
            elif match['stage'] == 'SF': iteration_team_ceilings[loser] = 'SF'
            elif match['stage'] == 'Final': 
                iteration_team_ceilings[loser] = 'Runner-up'
                iteration_team_ceilings[match['winner']] = 'Champion'
                
        for team, stage in iteration_team_ceilings.items():
            survival_matrix[team][stage] += 1
            
        total_over_6_games += iter_over_6_count
        if iter_over_6_count > max_over_6_in_single_iter:
            max_over_6_in_single_iter = iter_over_6_count
            
        total_upsets_count += len(iter_upset_list)
        if len(iter_upset_list) > max_upsets_in_single_iter:
            max_upsets_in_single_iter = len(iter_upset_list)
            peak_upset_iteration_index = iteration
            peak_iteration_upset_matches_list = iter_upset_list

    print("\n=======================================================")
    print("         REAL BRACKET: TEAM SURVIVAL MATRIX            ")
    print("=======================================================")
    survival_df = pd.DataFrame.from_dict(survival_matrix, orient='index')
    columns_order = ['R32', 'R16', 'QF', 'SF', 'Runner-up', 'Champion']
    survival_df = survival_df[columns_order].sort_values(by='Champion', ascending=False)
    print(survival_df.to_string())
    
    print("\n=======================================================")
    print("            HIGH-SCORING THRILLER LOGS                 ")
    print("=======================================================")
    print(f"Total matches exceeding 6 goals: {total_over_6_games}")
    print(f"Max high-scoring matches in a single tournament: {max_over_6_in_single_iter}")
    print(f"Highest scoring single game recorded: {absolute_highest_scoring_game['match_details']}")

    print("\n=======================================================")
    print("             DYNAMIC BRACKET UPSETS                    ")
    print("=======================================================")
    print(f"Total model-driven upsets across all iterations: {total_upsets_count}")
    print(f"Most chaotic bracket iteration: #{peak_upset_iteration_index} (with {max_upsets_in_single_iter} upsets)")
    print("\nUpset match logs from that specific chaotic run:")
    for log in peak_iteration_upset_matches_list:
        print(f" - {log}")
        
    return survival_df

In [81]:
print("Starting execution call...")
knockout_results = simulate_knockouts(real_world_matchups, iterations=1000)

Starting execution call...

         REAL BRACKET: TEAM SURVIVAL MATRIX            
                        R32  R16   QF   SF  Runner-up  Champion
Spain                   382  277  135   87         56        63
Argentina               338  276  131  118         74        63
England                 407  246  166   78         47        56
Morocco                 460  221  150   75         42        52
Belgium                 532  182  136   62         37        51
Germany                 358  297  163   94         38        50
Mexico                  410  298  126   77         39        50
Senegal                 468  208  172   65         39        48
France                  340  281  170  122         39        48
Portugal                443  262  130   74         49        42
Japan                   478  255  125   61         40        41
Netherlands             540  205  117   65         35        38
Brazil                  522  216  135   52         38        37
Croatia             